# 1. Components

The diagram shows the objects of `shapes.project` and how they fit together. An arrow goes from an object to one it holds, labelled with the attribute, such as `project.sample_model`. The arrows from the library show what is loaded from it. The EasyShapes GUI app works on the same objects, on its Project, Sample Model and Analysis pages. Notebooks 1 to 3 use the parts on their own; notebook 4 puts them together in a project.

This notebook works with the components (such as Lipids and Surfactants), and the library they are loaded from and saved to.

```
                                   ┌─────────┐
                                   │ Project │
                                   └────┬────┘
               ┌────────────────────────┴────────────────────────┐
          sample_model                                        analysis
               ▼                                                 ▼
        ┌─────────────┐                                    ┌──────────┐
        │ SampleModel │                                    │ Analysis │
        └──────┬──────┘                                    └─────┬────┘
      ┌────────┴─────────┐                                       │
  structure            buffer                              equilibration
      │                  ▼                                       ▼
      │              ┌────────┐                          ┌───────────────┐
      │              │ Buffer │                          │ Equilibration │
      │              └───┬────┘                          └───────┬───────┘
      │           ┌──────┴──────┐                       ┌────────┴─────────┐
      │        solvent      components            configuration          stages
      │           ▼             ▼                       ▼                  ▼
      │      ┌─────────┐     ┌──────┐       ┌───────────────────────┐  ┌───────┐
      │      │ Solvent │     │ Salt │       │ AnalysisConfiguration │  │ Stage │
      │      └────▲────┘     └──┬───┘       └───────────▲───────────┘  └───────┘
      │        Library    cation, anion              Library
      │                         ▼
      │                      ┌─────┐
      │                      │ Ion │
      │                      └──▲──┘
    one of                   Library
      └────┬────────────────────────┬─────────────────────┐
           ▼                        ▼                     ▼
┌─────────────────────┐        ┌─────────┐        ┌────────────────┐
│ Ring / Rod /        │◄─motif─┤ Lattice ├─motif─►│ Ball / Vesicle │
│ Bilayer / Monolayer │        └─────────┘        └─┬───────────┬──┘
└──────────┬──────────┘                             │    layers / lamellae
           │                                        │           ▼
           │                                        │  ┌─────────────────┐
           │                                        │  │ Layer / Lamella │
           └───────────────────┬────────────────────┘  └─────────────────┘
                           components
                               ▼
                     ╔═══════════════════╗   load   ╔═════════╗
your file, SMILES ──►║ Component         ║◄─────────╢ Library ║
                     ║ Surfactant, Lipid ╟─────────►║         ║
                     ╚═════════╤═════════╝   save   ╚═════════╝
                          counter_ion
                               ▼
                            ╔═════╗
                            ║ Ion ║
                            ╚══▲══╝
                            Library
```

Components are the molecules that a structure is built from: lipids, surfactants, ions and solvents.

In this notebook you will:

1. see the assets in the Shapespyer library
2. load a component from the library and look at it
3. make a component from a file of your own
4. make a component from a SMILES string
5. save your components to the library, so every notebook can load them

We have to start with a few setup cells. On `Colab` it installs Shapespyer, which takes about a minute. If you run it locally, make sure to install it in your virtual environment. [Shapespyer installation docs](https://simnavi.gitlab.io/shapespyer.io/installation.html)

In [ ]:
# @title Setup: on Colab, install Shapespyer (about 1 min) { display-mode: "form" }
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
CONFIG_FILE = Path("shapespyer.yaml")  # where Shapespyer keeps its files

if IN_COLAB:
    %pip install -q "shapespyer==0.3.1"

    # nglview draws molecules through Colab's widget manager
    from google.colab import output
    output.enable_custom_widget_manager()
    # and without its frame player: one-frame views don't use it, and its jslink opens Colab's widget panel
    import nglview
    nglview.NGLWidget._create_player = lambda self: None

    # the library, Temp and Projects go to /content, to browse them in the Files panel
    CONFIG_FILE.write_text("app_path: /content/shapespyer\n")
elif not CONFIG_FILE.is_file():
    err_msg = f"No {CONFIG_FILE} in {Path.cwd()}; run the notebook in its own folder, next to shapespyer.yaml"
    raise FileNotFoundError(err_msg)

The next cell provides helper functions for us to use with Colab. It provides functions that only draw molecules, download an example file and check the library.
These helper functions, as well as the helpers in other notebooks, **are not part of the Shapespyer workflow**. They are all in the `ColabHelpers` class, and the notebook always calls them from it, as `ColabHelpers.<name>()`, so you can tell them apart from the Shapespyer package.

In [ ]:
# @title ColabHelpers (not part of the Shapespyer workflow) { display-mode: "form" }
import urllib.request

import mdtraj
import nglview


class ColabHelpers:
    """Workshop helpers, not part of Shapespyer: call them as ColabHelpers.<name>()."""

    EXAMPLE_FILES_URL = "https://gitlab.com/simnavi/shapespyer/-/raw/v0.3.1/assets/components"

    @staticmethod
    def show_structure(component, box=False):
        """Draw a component, as its .show() does; Shapespyer 0.3.1's .show() fails on Colab.

        box=True also draws the box of the structure file.
        """
        view = nglview.show_structure_file(str(component.structure_file_path))
        view.clear_representations()
        view.add_representation("ball+stick", selection=component.name, radius=0.25)
        if box:
            view.add_representation("unitcell")
        return view

    @staticmethod
    def download_example_file(path, folder):
        """Download a file of the Shapespyer repository into a folder, to stand in for a file of your own."""
        folder.mkdir(parents=True, exist_ok=True)
        target = folder / path.rsplit("/", 1)[-1]
        urllib.request.urlretrieve(f"{ColabHelpers.EXAMPLE_FILES_URL}/{path}", target)
        return target

    @staticmethod
    def show_component(component, box=False):
        """Draw only the residues named like the component, for a file that holds other molecules too.

        box=True also draws the box of the structure file.
        """
        frame = mdtraj.load(str(component.structure_file_path))
        atoms = frame.top.select(f"resname {component.name}")
        print(f"{component.name}: {len(atoms)} of the {frame.n_atoms} atoms in {component.structure_file_path.name}")
        view = nglview.show_mdtraj(frame.atom_slice(atoms))
        if box:
            view.add_representation("unitcell")
        return view

    @staticmethod
    def show_library_entry(library, asset_type, name):
        """Print whether you saved a library entry yourself, and its files."""
        record = library.find(asset_type, name=name)
        print(f"{name} saved by you:", "ser_data" in record)
        for path in record.get("paths", []):
            print(" ", path)

In the setup we pass the `shapespyer.yaml` config file, which says where Shapespyer keeps its files. On `Colab` it is necessary, but in the common scenario, users won't have to use the config file, as the `shapes.project` package manages files automatically. Use the config if you want manual control over where the files are kept.

In [ ]:
# @title Config file, log level and work folder { display-mode: "form" }
from shapes.basics.utils import LogConfiguration
from shapes.project.files import AppFileManager

# the first call sets where the files go, for the whole session
config = AppFileManager(CONFIG_FILE).config
print("Workshop files:", config.app_path)

# the level applies to Shapespyer's own handlers; stop the messages there, or Colab prints them all again
LogConfiguration("WARNING").logger.propagate = False

# the downloaded example file goes to a work folder, which the clean up deletes
WORK = config.app_path / "work"

In [4]:
from shapes.project import Ion, Library, Surfactant

### The library

Here you choose where the library keeps its files: both the assets downloaded from the Shapespyer assets and the components you save yourself. We give `LocalAssets/` in the application directory, which is also the default. On Colab that is `/content/shapespyer/LocalAssets`: open the Files panel on the left to see it.

The first `Library()` of a session reads the remote index of assets and downloads their files into that folder. Only that first call sets the path: later calls return the same library.

What you will use from the library:

- `available_assets`: the names of every asset available, local or remote, by asset type.
- `has_entry(asset_type, name)`: tells whether a name is taken in the library.
- `delete_entry(asset_type, name, force=False)`: deletes what the library stores for an entry. `force=True` deletes it even though assets of this session hold its files.
- `sync(all_remote=False, force=False)`: updates the local copies of remote assets to the remote index. `all_remote=True` also downloads the remote assets you have no copy of yet; `force=True` also replaces the assets you saved yourself with the remote ones.
- `clean_temp(force=False)`: deletes the dir, which holds the working files of assets and starts it afresh.
- `available_projects`, `available_examples` and `delete_project(path, force=False)`: the saved projects, the example projects, and deleting a project. Notebook 4 uses them.

Loading and saving go through the components themselves: `Surfactant.load(name)` and `.save()`, below.

These are the assets ready to use, by type:

In [6]:
library = Library(path=config.app_path / "LocalAssets")
library.available_assets

{'AnalysisConfiguration': ['CHARMM36'],
 'ForceField': ['CHARMM36'],
 'Ion': ['BRA', 'CLA', 'NAT'],
 'Lipid': ['CHL', 'DLPE', 'DOPC', 'DOPE', 'DPPC', 'DPPE', 'POPC', 'POPE'],
 'Solvent': ['ETOH', 'TIP3'],
 'Surfactant': ['CTA', 'SDS']}

### Load a component and look at it

`Surfactant.load(name)` takes a surfactant from the library; `Lipid`, `Ion` and `Solvent` load the same way. A surfactant, like a lipid, holds:

- `name`: the name of the component, also the residue name of its molecules in the structure file.
- `paths`: the files of the component: its structure file (`.gro` or `.pdb`) and its topology (`.itp`).
- `counter_ion`: the counter ion, or the name of one to load from the library. We provide one further.
- `mint` and `mext`: the indices of the interior and the exterior (back-)bone atoms, closer to and farther from the structure centre. A structure uses them to place the molecule; notebook 2 explains them in detail.

`ColabHelpers.show_structure()` draws the molecule, as the component's `.show()` does. Shapespyer 0.3.1's `.show()` fails on Colab. It must be the last line of the cell.

In [ ]:
sds = Surfactant.load("SDS")
ColabHelpers.show_structure(sds)

Surfactant(SDS) with
- counter_ion: None
- mext: 0
- mint: 0
- paths: (PosixPath('/home/ales/shapespyer-workshop/.data/LocalAssets/Surfactant/SDS/SDS.gro'), PosixPath('/home/ales/shapespyer-workshop/.data/LocalAssets/Surfactant/SDS/SDS.itp'))
- name: SDS


NGLWidget()

### Give a component its counter ion

In water, each SDS molecule comes with a sodium ion: SDS is sodium dodecyl sulfate. `counter_ion` says which ion goes with the component. The ion is loaded from the library, like the component: `available_assets["Ion"]` above lists the ions, and `NAT` is sodium. When a sample model is assembled, the counter ions go into the box with the component; notebook 4 does that.

In [8]:
sds.counter_ion = Ion.load("NAT")
sds.counter_ion.paths

(PosixPath('/home/ales/shapespyer-workshop/.data/LocalAssets/Ion/NAT/NAT.gro'),
 PosixPath('/home/ales/shapespyer-workshop/.data/LocalAssets/Ion/NAT/NAT.itp'))

### A component from your own file

Here `ColabHelpers.download_example_file()` downloads a `.gro` file from the Shapespyer repository, as if it were your own. The name of the component must match the residue name in the file. This file also holds an SDS molecule next to the CTA, and `ColabHelpers.show_structure()` would draw both, so `ColabHelpers.show_component()` draws only the residues named CTA.

On Colab you can also upload your own file: drag it into the Files panel, or run `from google.colab import files; files.upload()`.

In [9]:
my_file = ColabHelpers.download_example_file("surfactants/CTA/CTA.gro", WORK)

cta = Surfactant(name="CTA", paths=[my_file])
ColabHelpers.show_component(cta)

CTA: 44 of the 86 atoms in CTA.gro


NGLWidget()

### A component from a SMILES string

Here we make SDS from its SMILES string. Hydrogens are added, and the atoms follow the SMILES order. We write the tail first, so the "bone" of the molecule runs from the tail end (`mint` is 0) to the head (`mext` is the last atom), as a micelle needs.

The name becomes the residue name in the file, at most 5 characters. We call it `SDSsm`, so it doesn't clash with the library's SDS. A structure finds its molecules by this residue name, so a component can't be renamed later. The component gets a structure file only, with no topology. So it can be built into structures, but not equilibrated.

`from_smiles` takes:

- `name`: the name of the component, also the residue name in the file, at most 5 characters.
- `smiles`: the SMILES string of one molecule, of two atoms or more.
- `box`: the box lengths along X, Y and Z of the written file, in nm. 3 nm on each side is more than one SDS molecule is long. `box=True` makes the helper draw the box.
- `formula`: the chemical formula for the title line of the file. Without it, the name is used.
- `cis`: kinks C=C double bonds as cis instead of trans. SDS has none.
- `align_z`: rotates the molecule so a terminal bond lies along Z.
- `flatten_xz`: flattens the aligned molecule into the XZ plane; it works only with `align_z`.
- `extension`: the format of the structure file, `.gro` or `.pdb`.

In [ ]:
sds_from_smiles = Surfactant.from_smiles(
    name="SDSsm",
    smiles="CCCCCCCCCCCCOS(=O)(=O)[O-]",
    box=(3.0, 3.0, 3.0),  # in nm
    formula="C12H25SO4",
    cis=False,
    align_z=True,
    flatten_xz=False,
    extension=".gro",
)
print(sds_from_smiles.number_of_atoms, sds_from_smiles.mint, sds_from_smiles.mext)
ColabHelpers.show_structure(sds_from_smiles, box=True)

### Save your components to the library

`.save()` copies the files of a component into the library, `LocalAssets/<type>/<name>/`, with a `ser_data.yaml` record of the whole component. From then on, `Surfactant.load(name)` finds it: in this notebook, in the other notebooks and in later sessions.

We save our two components:

- `SDSsm` has a new name, so it is simply added.
- `CTA` has the name of a library asset, so saving it replaces that asset in your local library.

If you run this notebook a second time without the clean up at the end, use `overwrite=True` here too.

In [ ]:
sds_from_smiles.save()
library.available_assets["Surfactant"]

The name `CTA` is taken by the library's CTA, so a plain `.save()` refuses:

In [ ]:
ColabHelpers.show_library_entry(library, "Surfactant", "CTA")
try:
    cta.save()
except ValueError as err:
    print(err)

`overwrite=True` replaces it. The library looks at your local entries first, and `sync()` leaves the entries you saved alone, so every notebook now loads your CTA. Here your file is the same as the library's; with a CTA of your own, such as one with other coordinates, yours would be used.

In [ ]:
cta.save(overwrite=True)
ColabHelpers.show_library_entry(library, "Surfactant", "CTA")

Load a saved component back from the library, as any other notebook would. It works on the copies in the library, not on the files it was made from:

In [ ]:
loaded = Surfactant.load("SDSsm")
print(loaded.paths)
ColabHelpers.show_structure(loaded)

### Clean up (optional)

This deletes the work folder with the downloaded `CTA.gro`, and your two library entries. Without this, every later notebook and session would load your CTA instead of the library's. `force=True` is needed, because `loaded` still works on the files of its entry. The library's CTA comes back with the next lookup, which downloads it again.

In [ ]:
import shutil

shutil.rmtree(WORK, ignore_errors=True)
library.delete_entry("Surfactant", "SDSsm", force=True)
library.delete_entry("Surfactant", "CTA", force=True)

ColabHelpers.show_library_entry(library, "Surfactant", "CTA")
print(WORK.exists(), library.available_assets["Surfactant"])